In [45]:
import pandas as pd
import numpy as np
import kagglehub
import re




from astroquery.gaia import Gaia
import requests
from bs4 import BeautifulSoup
import json

import datetime

import os

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#defining global variable path to files and data, it is cvalled thropughout modify this path to match your local path
path = 'E:\\GG\\Msc\\DATA9910 (23115C)_working_with_data_lucas_rizzo_10ects_Thurs\\github\\wwd_9910_thurs\\labs and exercises\\ca assignment-4-due-wk13\\working'

## Project Objective

Project Question = *Does the Sun have a twin or a closely related cousin?*

Yhe goal of this project is to see can we determine if there another star in the universe that is identical or similar to our Sun, that we know of and have measurements, metrics for.

We will tak astronmical data from avariety of diffeent public sources.  

#### 1) Dataset - NASA >  Stellar Hosts

Data set taken from https://exoplanetarchive.ipac.caltech.edu/ 

We will down lai the 'Stellar Hosts' dataset. This dataset is 1 row per star. It is determined by the search for exoplanets, sok each star will have 0 to > 0 number of planets orbitting it.

In [8]:
# after a bit of playing around tihe the url format we get the url for csv format for the entire stellar hosts dataset
# we download in csv format
# This uses their preferred TAP API, like a sql command to directly query the database, and we get it in a clean csv.
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+stellarhosts&format=csv"
stellar_host_df = pd.read_csv(url)

stellar_host_df.shape

#df.head()

# NOTE th is nto one row per star >> all sceintific measurements are kept and sources are different, so needs to be filered....
# may add update a paramater, i.e. update mass of a star , added in a new row, old row kept...

(46870, 136)

In [9]:
# number of planets per star ie.. 1, 2, 3, 4 planets and how many stars have that number of planets
stellar_host_df['sy_pnum'].value_counts()

# for us we may wnay onyl jsuns which have at least 1 planet to find a match with our Sun?

sy_pnum
1    29489
2     9675
3     4367
4     2150
5      857
6      244
8       49
7       39
Name: count, dtype: int64

In [10]:
# number of unique star names in the dataset?
stellar_host_df['hostname'].nunique()

4969

In [13]:
cols_list = stellar_host_df.columns.tolist()
print(len(cols_list))
print(cols_list)

136
['hostname', 'hd_name', 'hip_name', 'tic_id', 'st_refname', 'sy_refname', 'ra', 'rastr', 'dec', 'decstr', 'glon', 'glat', 'elon', 'elat', 'sy_icmag', 'sy_icmagerr1', 'sy_icmagerr2', 'st_teff', 'st_tefferr1', 'st_tefferr2', 'st_tefflim', 'st_met', 'st_meterr1', 'st_meterr2', 'st_metlim', 'st_radv', 'st_radverr1', 'st_radverr2', 'st_radvlim', 'st_vsin', 'st_vsinerr1', 'st_vsinerr2', 'st_vsinlim', 'st_lum', 'st_lumerr1', 'st_lumerr2', 'st_lumlim', 'st_logg', 'st_loggerr1', 'st_loggerr2', 'st_logglim', 'st_age', 'st_ageerr1', 'st_ageerr2', 'st_agelim', 'st_mass', 'st_masserr1', 'st_masserr2', 'st_masslim', 'st_dens', 'st_denserr1', 'st_denserr2', 'st_denslim', 'st_rad', 'st_raderr1', 'st_raderr2', 'st_radlim', 'sy_snum', 'sy_pnum', 'sy_mnum', 'sy_pm', 'sy_pmerr1', 'sy_pmerr2', 'sy_pmra', 'sy_pmraerr1', 'sy_pmraerr2', 'sy_pmdec', 'sy_pmdecerr1', 'sy_pmdecerr2', 'sy_plx', 'sy_plxerr1', 'sy_plxerr2', 'sy_dist', 'sy_disterr1', 'sy_disterr2', 'sy_bmag', 'sy_bmagerr1', 'sy_bmagerr2', 'sy_vma

In [ ]:
# column mames are abbreviatinos and not very intuitive or human readable
# as such we will  craete a function to scrape the web page for the table column and label anmes
# we want to retain any units and case with thsoe, 
# https://stackoverflow.com/questions/48071598/parsing-through-html-with-beautifulsoup-in-python
def map_col_label_names_dict(url):
    """
    Scrapes the NASA Exoplanet Archive documentation for column definitions, returning Database Column Name, Table Label (lowercase, underscores), and Description. The original Table Label is omitted.
    """

    response = requests.get(url, timeout=10)
    #response.raise_for_status()
    html_content = response.text
    
    #Parse the HTML
    soup = BeautifulSoup(html_content, 'html.parser')
    # craete a list to store the column values
    column_defs = []
    # find all the tables present on the page
    tables = soup.find_all('table')

   #for eery table on the page, find the rows
    for table in tables:
        # tr are the tble rows as per the html source >>  <tr class="column" id="tic_id">
        rows = table.find_all('tr')
        
        # whenw e sinspect the html code we see > <td class="label">hostname&dagger;</td>
        # h are the tabel headers >> <th width="99">Database Column Name</th>
        for row in rows:
            cells = row.find_all(['td', 'th'])
            
            # Need at least 3 cells (Name, Label, Description)
            if len(cells) >= 3:
                db_col_name = cells[0].get_text(strip=True)
                table_label = cells[1].get_text(strip=True)
                description = cells[2].get_text(strip=True)
                
                # Skip header/title rows
                if db_col_name.lower() in ["database column name", "uncertainties column", "table label"]:
                    continue
                   
                #tofy the database column name to remove stray characters
                #cleaned_database_col_name = re.sub(r'\W+', '', database_col_name.replace('†', ''))
                
                # create the Clean_Label (lowercase + units/spaces to underscore)
                clean_label = table_label.lower()
                clean_label = re.sub(r'[^a-z0-9_]+', '_', clean_label)
                clean_label = re.sub(r'_+', '_', clean_label).strip('_')

                if db_col_name:
                    column_defs.append({
                        'db_col_name': db_col_name,
                        'label': clean_label, # Renamed key to 'Label' for simplicity
                        'description': description
                    })

    return column_defs

# URL for the Stellar Hosts documentation
url = "https://exoplanetarchive.ipac.caltech.edu/docs/API_STELLARHOSTS_columns.html"

# Execute the function
stellarhosts_defs = map_col_label_names_dict(url)

print("--- Example of the Simplified List of Dictionaries Output (First 3 Records) ---")
print(stellarhosts_defs[:3])

--- Example of the Simplified List of Dictionaries Output (First 3 Records) ---
[{'db_col_name': 'sy_name†', 'label': 'system_name', 'description': 'Name of the system star is in'}, {'db_col_name': 'hostname†', 'label': 'host_name', 'description': 'Stellar name most commonly used in the literature'}, {'db_col_name': 'hd_name', 'label': 'hd_id', 'description': 'Name of the star as given by the Henry Draper Catalog'}]


In [46]:
df_defs = pd.DataFrame(stellarhosts_defs)
#df_defs.shape
#print(df_defs.head(30))

gaia_defs = [item for item in stellarhosts_defs if 'gaia' in item.get('db_col_name', "").lower()]

print(json.dumps(gaia_defs,  indent=4))

[
    {
        "db_col_name": "gaia_dr2_id",
        "label": "gaia_dr2_id",
        "description": "Name of the star as given by the Gaia DR2 Catalog"
    },
    {
        "db_col_name": "gaia_dr3_id",
        "label": "gaia_dr3_id",
        "description": "Name of the star as given by the Gaia DR3 Catalog"
    },
    {
        "db_col_name": "sy_gaiamag\u2020",
        "label": "gaia_magnitude",
        "description": "Brightness of the host star as measuring using the \nGaia band in units of magnitudes. Objects matched \nto Gaia using the Hipparcos or 2MASS IDs provided \nin Gaia DR2."
    }
]


Need to get a list of GAIA IDs from this dataframe, as this will be used to get select the correct row data from other datasets

In [39]:
# get gaia columns from the datafram
gaia_cols = [col for col in stellar_host_df.columns if'gaia' in col.lower()]

print(f"gaia columns  : {gaia_cols}")

gaia columns  : ['sy_gaiamag', 'sy_gaiamagerr1', 'sy_gaiamagerr2', 'gaia_dr2_id', 'gaia_dr3_id']


In [ ]:
# stellar_host_df.columns
# how many not na in both
stellar_host_df[['gaia_dr2_id', 'gaia_dr3_id']].notna().sum()


gaia_dr2_id    45445
gaia_dr3_id    45132
dtype: int64

In [55]:
stellar_host_df_2 = stellar_host_df.copy()

stellar_host_df_2['dr2_num'] = stellar_host_df_2['gaia_dr2_id'].str.extract(r'(\d+)', expand=False)
stellar_host_df_2['dr3_num'] = stellar_host_df_2['gaia_dr3_id'].str.extract(r'(\d+)', expand=False)

stellar_host_df_2['dr2_equals_dr3'] = stellar_host_df_2['dr2_num'] == stellar_host_df_2['dr3_num']

stellar_host_df_2['dr2_equals_dr3'].value_counts()

dr2_equals_dr3
False    46870
Name: count, dtype: int64

In [54]:
stellar_host_df['dr2_equals_dr3'] = (stellar_host_df['gaia_dr2_id'] == stellar_host_df['gaia_dr3_id'])

stellar_host_df.head()
#stellar_host_df['dr2_equals_dr3'].value_counts()

,hostname,hd_name,hip_name,tic_id,st_refname,sy_refname,ra,rastr,dec,decstr,...,sy_kepmagerr1,sy_kepmagerr2,st_rotp,st_rotperr1,st_rotperr2,st_rotplim,gaia_dr2_id,gaia_dr3_id,cb_flag,dr2_equals_dr3
0,Kepler-936,NaN,NaN,TIC 270619059,<a refstr=Q1_Q17_DR25_KOI_TABLE href=https://e...,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,293.577796,19h34m18.67s,46.736295,+46d44m10.66s,...,NaN,NaN,NaN,NaN,NaN,NaN,Gaia DR2 2128190453050802048,Gaia DR3 2128190453050802048,0,False
1,Kepler-55,NaN,NaN,TIC 164884002,<a refstr=Q1_Q12_KOI_TABLE href=https://exopla...,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,285.168329,19h00m40.40s,44.026490,+44d01m35.36s,...,NaN,NaN,NaN,NaN,NaN,NaN,Gaia DR2 2105930840143687680,Gaia DR3 2105930840143687680,0,False
2,Kepler-948,NaN,NaN,TIC 138099276,<a refstr=Q1_Q17_DR25_KOI_TABLE href=https://e...,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,293.300277,19h33m12.07s,41.523324,+41d31m23.97s,...,NaN,NaN,NaN,NaN,NaN,NaN,Gaia DR2 2077595394707557120,Gaia DR3 2077595394707557120,0,False
3,Kepler-917,NaN,NaN,TIC 268383727,<a refstr=Q1_Q8_KOI_TABLE href=https://exoplan...,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,299.051256,19h56m12.30s,47.949268,+47d56m57.36s,...,NaN,NaN,NaN,NaN,NaN,NaN,Gaia DR2 2085724496490595584,Gaia DR3 2085724496490595584,0,False
4,Kepler-657,NaN,NaN,TIC 159643991,<a refstr=Q1_Q6_KOI_TABLE href=https://exoplan...,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,291.197301,19h24m47.35s,47.307755,+47d18m27.92s,...,NaN,NaN,NaN,NaN,NaN,NaN,Gaia DR2 2129158435598210816,Gaia DR3 2129158435598210816,0,False


In [ ]:
canonical_map = {
    "gaia_source_id": {
        "aliases": ["gaia_id", "gid", "source_id", "dr3_source_id"],
        "description": "Gaia DR3 unique source identifier",
        "label": "Gaia DR3 Source ID",
        "dtype": "string",
        "units": None
    },
    "ra": {
        "aliases": ["ra", "raj2000", "ra_deg"],
        "description": "Right ascension in ICRS (degrees)",
        "label": "RA",
        "dtype": "float64",
        "units": "deg"
    },
    "dec": {
        "aliases": ["dec", "dej2000", "dec_deg"],
        "description": "Declination in ICRS (degrees)",
        "label": "DEC",
        "dtype": "float64",
        "units": "deg"
    },
    "teff": {
        "aliases": ["st_teff", "teff_gspphot", "lamost_teff"],
        "description": "Stellar effective temperature (Kelvin)",
        "label": "Teff",
        "dtype": "float64",
        "units": "K"
    },
    "logg": {
        "aliases": ["st_logg", "logg_gspphot", "lamost_logg"],
        "description": "Surface gravity log10(cm s-2)",
        "label": "log(g)",
        "dtype": "float64",
        "units": "dex"
    },
    "feh": {
        "aliases": ["st_met", "mh_gspphot", "feh_lamost"],
        "description": "Metallicity [Fe/H] relative to the Sun",
        "label": "Fe/H",
        "dtype": "float64",
        "units": "dex"
    }
}
